# Notebook 04: FSDP and ZeRO Sharding

## Why This Matters

DDP replicates the full model on every GPU. For a 7B model in float32, that is 28 GB -- more than an A100 80GB can hold once you account for activations and optimizer states. ZeRO (Zero Redundancy Optimizer) and FSDP (Fully Sharded Data Parallel) solve this by **sharding** parameters, gradients, and optimizer states across GPUs. FSDP is how Meta trains LLaMA, and it is the standard approach for models that do not fit on a single GPU.


In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

n_gpus = torch.cuda.device_count()
print(f"Available GPUs: {n_gpus}")
print(f"PyTorch version: {torch.__version__}")
print("Ready.")

## 1. ZeRO: Three Stages of Memory Reduction

ZeRO (Zero Redundancy Optimizer, Rajbhandari et al. 2019) progressively shards different components:

**ZeRO Stage 1**: Shard **optimizer states** only (Adam: first and second moments)
- Memory: optimizer states consume 12 bytes/param (for Adam: 4 bytes gradient + 4 bytes m1 + 4 bytes m2)
- Savings: 4x reduction in optimizer state memory (from 12 bytes to 3 bytes per GPU in 4-GPU setup)

**ZeRO Stage 2**: Shard optimizer states + **gradients**
- Savings: additional factor of N for gradient memory
- Total: ~8x memory reduction for Adam with 8 GPUs

**ZeRO Stage 3**: Shard optimizer states + gradients + **parameters**
- Every parameter is stored on only 1 GPU
- Total: ~Nx reduction vs DDP (ideal: perfect linear scaling)
- Cost: need all-gather before each forward pass

FSDP = ZeRO-3 implemented in PyTorch.


In [ ]:
# Memory analysis: DDP vs ZeRO stages
def memory_per_gpu_bytes(n_params, n_gpus, zero_stage=0, dtype_bytes=2):
    # dtype_bytes: 2 for bfloat16, 4 for float32
    # Parameters
    if zero_stage >= 3:
        params = n_params * dtype_bytes / n_gpus
    else:
        params = n_params * dtype_bytes
    
    # Gradients
    if zero_stage >= 2:
        grads = n_params * dtype_bytes / n_gpus
    else:
        grads = n_params * dtype_bytes
    
    # Optimizer states (Adam: fp32 master weights + m1 + m2 = 12 bytes/param)
    if zero_stage >= 1:
        optim_states = n_params * 12 / n_gpus
    else:
        optim_states = n_params * 12
    
    return (params + grads + optim_states) / 1e9  # GB

# Analysis for a 7B parameter model
n_params = 7e9
n_gpus_list = [1, 2, 4, 8, 16, 32, 64]

fig, ax = plt.subplots(figsize=(10, 6))

for stage, label, color in [
    (0, 'DDP (ZeRO-0)', 'red'),
    (1, 'ZeRO Stage 1', 'orange'),
    (2, 'ZeRO Stage 2', 'green'),
    (3, 'ZeRO Stage 3 / FSDP', 'blue'),
]:
    mem = [memory_per_gpu_bytes(n_params, n, zero_stage=stage) for n in n_gpus_list]
    ax.semilogy(n_gpus_list, mem, 'o-', label=label, color=color, linewidth=2)

ax.axhline(y=80, color='black', linestyle='--', alpha=0.5, label='A100 80GB limit')
ax.axhline(y=40, color='gray', linestyle='--', alpha=0.5, label='A100 40GB limit')
ax.set_xlabel('Number of GPUs')
ax.set_ylabel('Memory per GPU (GB)')
ax.set_title('7B Parameter Model: Memory per GPU by ZeRO Stage')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xticks(n_gpus_list)

plt.tight_layout()
plt.savefig('zero_memory.png', dpi=100)
plt.show()

print('Memory per GPU for 7B model (bfloat16, Adam):')
print(f'{'Stage':<25} | {8} GPUs')
print('-' * 40)
for stage, label in [(0,'DDP'), (1,'ZeRO-1'), (2,'ZeRO-2'), (3,'FSDP/ZeRO-3')]:
    mem = memory_per_gpu_bytes(n_params, 8, zero_stage=stage)
    fits = 'fits' if mem < 80 else 'OOM'
    print(f'{label:<25} | {mem:.1f} GB ({fits} on A100 80GB)')

## 2. FSDP Forward and Backward Pass

FSDP changes the computation pattern:

**Forward pass**:
1. Before each FSDP module's forward: **all-gather** its parameters from all ranks
2. Run the module forward
3. Free the gathered parameters (to save memory)

**Backward pass**:
1. Before each FSDP module's backward: **all-gather** its parameters again
2. Compute gradients
3. **Reduce-scatter** gradients -- each rank keeps only its shard
4. Free gathered parameters again

**Optimizer step**:
1. Each rank updates only its parameter shard (using its local optimizer state)

This means parameters are transferred twice per forward pass (forward + backward) instead of once (DDP). The memory savings must outweigh the extra communication.


In [ ]:
# FSDP communication pattern analysis

def fsdp_comm_volume(n_params, n_gpus, dtype_bytes=2):
    param_bytes = n_params * dtype_bytes
    # Forward: all-gather (N->N) = P bytes per GPU
    # Backward: all-gather (P bytes per GPU) + reduce-scatter (P bytes per GPU)
    per_gpu = 3 * param_bytes  # 3 all-gathers effectively
    # Ring bandwidth: 2*(N-1)/N * P
    actual_per_gpu = 2 * (n_gpus - 1) / n_gpus * param_bytes * 3
    return actual_per_gpu / 1e9

def ddp_comm_volume(n_params, n_gpus, dtype_bytes=2):
    param_bytes = n_params * dtype_bytes
    # One all-reduce = 2*(N-1)/N * P
    return 2 * (n_gpus - 1) / n_gpus * param_bytes / 1e9

n_params = 7e9
n_gpus = 8

ddp_vol = ddp_comm_volume(n_params, n_gpus)
fsdp_vol = fsdp_comm_volume(n_params, n_gpus)

print('Communication volume per step (7B model, 8 GPUs, bfloat16):')
print(f'DDP all-reduce:          {ddp_vol:.1f} GB')
print(f'FSDP all-gather + r-s:   {fsdp_vol:.1f} GB')
print(f'FSDP overhead vs DDP:    {fsdp_vol/ddp_vol:.1f}x more communication')
print()
print('The tradeoff: FSDP communicates more, but uses N x less memory.')
print('For models that fit with DDP -> use DDP.')
print('For models that require FSDP -> accept the communication overhead.')

## 3. FSDP API: Wrapping Policy and Sharding Strategy

The key FSDP configuration decisions:

1. **Wrapping policy**: which modules to wrap with FSDP. The standard for transformers: `transformer_auto_wrap_policy` wraps each transformer layer independently.

2. **Sharding strategy**:
   - `FULL_SHARD` (ZeRO-3): params + grads + optimizer states all sharded
   - `SHARD_GRAD_OP` (ZeRO-2): params unsharded during forward, grads + optimizer sharded
   - `HYBRID_SHARD`: shard within a node, replicate across nodes

3. **CPU offload**: move optimizer states (and optionally parameters) to CPU RAM. Saves GPU memory but slows training by ~20-50% (PCIe bandwidth bottleneck).


In [ ]:
# FSDP API demonstration (structure only -- requires multi-GPU to fully run)
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, CPUOffload, MixedPrecision
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from functools import partial

# Example transformer model
class TransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn_q = nn.Linear(d_model, d_model)
        self.attn_k = nn.Linear(d_model, d_model)
        self.attn_v = nn.Linear(d_model, d_model)
        self.attn_o = nn.Linear(d_model, d_model)
        self.ff1 = nn.Linear(d_model, d_ff)
        self.ff2 = nn.Linear(d_ff, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        return x  # simplified

class GPTModel(nn.Module):
    def __init__(self, n_layers=24, d_model=1024, n_heads=16, d_ff=4096):
        super().__init__()
        self.embed = nn.Embedding(50000, d_model)
        self.layers = nn.ModuleList([
            TransformerLayer(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, 50000, bias=False)
    
    def forward(self, x):
        return self.embed(x)

model = GPTModel(n_layers=4, d_model=512)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params/1e6:.1f}M')

print('\nFSDP wrapping code (requires distributed init to actually run):')
fsdp_code = '''
# Wrapping policy: each TransformerLayer gets its own FSDP unit
auto_wrap_policy = partial(
    transformer_auto_wrap_policy,
    transformer_layer_cls={TransformerLayer}
)

# Mixed precision: compute in bfloat16, reduce in float32
mixed_precision_policy = MixedPrecision(
    param_dtype=torch.bfloat16,
    reduce_dtype=torch.float32,
    buffer_dtype=torch.float32,
)

# FSDP wrapping
model = FSDP(
    model,
    auto_wrap_policy=auto_wrap_policy,
    sharding_strategy=ShardingStrategy.FULL_SHARD,  # ZeRO-3
    mixed_precision=mixed_precision_policy,
    cpu_offload=CPUOffload(offload_params=False),   # keep on GPU
    device_id=torch.device(f"cuda:{local_rank}"),
)
'''
print(fsdp_code)

## 4. Memory Math: FSDP vs DDP for a 7B Model

For a 7B parameter model in bfloat16 on 8 x A100 80GB:

| Component | DDP per GPU | FSDP per GPU |
|-----------|-------------|--------------|
| Parameters (bf16) | 14 GB | 1.75 GB |
| Gradients (bf16) | 14 GB | 1.75 GB |
| Optimizer states (fp32) | 56 GB | 7 GB |
| **Total** | **84 GB** | **10.5 GB** |

DDP: 84 GB -- **does not fit** on 80 GB A100.
FSDP: 10.5 GB -- leaves ~69 GB for activations and batch.

With FSDP you can fit roughly 7x larger batches or 7x more layers. This is why FSDP is essential for LLM training.


## Summary

### Key Concepts

| ZeRO Stage | Shards | Memory reduction (8 GPUs) | Extra comm vs DDP |
|-----------|--------|--------------------------|-------------------|
| 0 (DDP) | Nothing | 1x | baseline |
| 1 | Optimizer states | ~2x | none |
| 2 | + Gradients | ~4x | none |
| 3 (FSDP) | + Parameters | ~8x | 3x more |

**Key rule**: Use DDP when your model fits. Use FSDP when it doesn't. The extra communication overhead of FSDP is worth it if it enables training at all.

**FSDP2** (PyTorch 2.1+): uses DTensor (distributed tensor) for cleaner abstractions, better support for HYBRID_SHARD, and improved CPU offloading.

**Next**: `05_ray_core_fundamentals.ipynb` -- distributed Python computation with Ray.
